In [1]:
import numpy as np
from numba import njit
from scipy.optimize import differential_evolution
from scipy.stats import linregress
import time
import mpmath
import multiprocessing

print("="*85)
print("🚀 算力怪兽觉醒：256核并行 | 单侧正谱线性拟合 (揭露基态漂移灾难)")
print("="*85)

if __name__ == '__main__':
    try:
        multiprocessing.set_start_method('fork', force=True)
    except RuntimeError:
        pass

# ==========================================
# 1. 精确黎曼零点
# ==========================================
def get_exact_riemann_zeros(n_max=150):
    mpmath.mp.dps = 15
    return np.array([float(mpmath.zetazero(i).imag) for i in range(1, n_max + 1)])

# ==========================================
# 2. 高性能内核：100亿步绝热冷却
# ==========================================
@njit(fastmath=True, nogil=True)
def build_ulam_matrix_anchored(u_temp, k_opt, steps, n_bins, offset):
    x = 0.5
    counts = np.zeros((n_bins, n_bins), dtype=np.float64)
    last_bin = int((x + 1.0) / 2.0 * (n_bins - 1))
    
    warmup_steps = 2000000 
    for i in range(warmup_steps):
        L_i = np.log(i + offset)
        u_dyn = u_temp + k_opt / (L_i**2)
        x = 1.0 - u_dyn * x**2
        if x > 1.0: x = 0.999
        elif x < -1.0: x = -0.999
            
    for i in range(warmup_steps, steps + warmup_steps):
        L_i = np.log(i + offset)
        u_dyn = u_temp + k_opt / (L_i**2)
        x = 1.0 - u_dyn * x**2
        if x > 1.0: x = 0.999
        elif x < -1.0: x = -0.999
        
        current_bin = int((x + 1.0) / 2.0 * (n_bins - 1))
        if 0 <= current_bin < n_bins and 0 <= last_bin < n_bins:
            counts[last_bin, current_bin] += 1
        last_bin = current_bin
        
    return counts

# ==========================================
# 3. 提取特征相位 (只保留正能量)
# ==========================================
def extract_phases(counts):
    row_sums = counts.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    P = counts / row_sums

    vals, _ = np.linalg.eig(P)
    # 🔥 这里我们按照传统实验，只看正半轴激发态
    valid_pos = vals[(vals.imag > 1e-5)]
    return np.unwrap(np.sort(np.angle(valid_pos)))

# ==========================================
# 4. 目标函数：正半轴自由线性拟合 (允许 b != 0)
# ==========================================
def objective_k_pos_linregress(params, target_zeros, u_c, steps, n_bins, offset):
    k_opt = params[0]  
    t_start_time = time.time()
    
    t_end = 1.0 / (np.log(steps + offset)**2)
    u_temp = u_c - k_opt * t_end  
    
    # 硬刚 100 亿步
    counts = build_ulam_matrix_anchored(u_temp, k_opt, steps, n_bins, offset)
    sys_phases = extract_phases(counts)
    
    N_compare = min(len(target_zeros), len(sys_phases))
    if N_compare < 80:
        return 1e6
        
    # 🔥 核心大戏：仅用正半轴进行全局回归，看看算法会不会通过产生截距 b 来"作弊"
    slope, intercept, r_value, p_value, std_err = linregress(
        sys_phases[:N_compare], target_zeros[:N_compare]
    )
    
    predicted_zeros = sys_phases[:N_compare] * slope + intercept
    error = np.mean((predicted_zeros - target_zeros[:N_compare])**2)
    
    print(f"[Worker] k={k_opt:.4f} | 仅正谱 MSE={error:.2f} (截距 b={intercept:.3f}) | Slope={slope:.2f}")
    return error

# ==========================================
# 5. 主程序：启动 256核 阵列
# ==========================================
if __name__ == '__main__':
    u_c = 1.543689
    scan_steps = 10_000_000_000  # 100 亿步
    scan_offset = 100000.0       
    scan_n_bins = 2000           
    
    print("[*] 正在加载靶向黎曼零点...")
    true_zeros = get_exact_riemann_zeros(100)
    
    print(f"[*] 演化步数已锁定为: 100亿步")
    print(f"[*] 寻优法则: 单侧正谱线性拟合 (暴露虚假漂移 b)")
    print(f"[*] ⚠️ 正在为 256核 阵列分配任务，起飞！")
    
    t_total = time.time()
    
    res = differential_evolution(
        func=objective_k_pos_linregress,
        bounds=[(2.0, 15.0)],    
        args=(true_zeros, u_c, scan_steps, scan_n_bins, scan_offset),
        strategy='best1bin',
        maxiter=10,             
        popsize=250,            
        tol=0.01,
        polish=False,           
        workers=-1,             
        disp=True
    )
    
    print(f"\n[+] 256核暴风寻优结束！总耗时: {(time.time()-t_total)/60:.2f} 分钟")
    if res.success or True: 
        best_k = res.x[0]
        
        print("\n" + "="*50)
        print(f"🎯 单侧谱截断灾难 验证完成：")
        print(f"[*] 最优参数 (k1) = {best_k:.5f}")
        print(f"[*] 此时算法妥协出的最小 MSE = {res.fun:.4f}")
        print("="*50)

🚀 算力怪兽觉醒：256核并行 | 单侧正谱线性拟合 (揭露基态漂移灾难)
[*] 正在加载靶向黎曼零点...
[*] 演化步数已锁定为: 100亿步
[*] 寻优法则: 单侧正谱线性拟合 (暴露虚假漂移 b)
[*] ⚠️ 正在为 256核 阵列分配任务，起飞！


/root/miniconda3/lib/python3.12/site-packages/scipy/optimize/_differentialevolution.py:518: UserWarning: differential_evolution: the 'workers' keyword has overridden updating='immediate' to updating='deferred'
  with DifferentialEvolutionSolver(func, bounds, args=args,


[Worker] k=6.2586 | 仅正谱 MSE=10.00 (截距 b=18.689) | Slope=290.08
[Worker] k=2.5859 | 仅正谱 MSE=35.74 (截距 b=15.937) | Slope=290.64
[Worker] k=8.3740 | 仅正谱 MSE=29.63 (截距 b=21.367) | Slope=291.77
[Worker] k=9.0661 | 仅正谱 MSE=24.98 (截距 b=18.302) | Slope=290.10
[Worker] k=4.9750 | 仅正谱 MSE=20.95 (截距 b=14.152) | Slope=291.71
[Worker] k=5.0266 | 仅正谱 MSE=27.11 (截距 b=18.104) | Slope=293.82
[Worker] k=6.6523 | 仅正谱 MSE=16.25 (截距 b=5.663) | Slope=306.28
[Worker] k=6.0648 | 仅正谱 MSE=25.24 (截距 b=12.409) | Slope=294.51
[Worker] k=6.7302 | 仅正谱 MSE=15.35 (截距 b=16.227) | Slope=287.32
[Worker] k=2.2733 | 仅正谱 MSE=9.02 (截距 b=8.618) | Slope=299.58
[Worker] k=3.6609 | 仅正谱 MSE=27.01 (截距 b=10.268) | Slope=297.87
[Worker] k=8.1681 | 仅正谱 MSE=24.74 (截距 b=14.351) | Slope=295.88
[Worker] k=11.4093 | 仅正谱 MSE=16.29 (截距 b=18.039) | Slope=297.12
[Worker] k=10.5892 | 仅正谱 MSE=17.34 (截距 b=11.322) | Slope=297.17
[Worker] k=6.2922 | 仅正谱 MSE=15.94 (截距 b=7.114) | Slope=294.31
[Worker] k=14.5618 | 仅正谱 MSE=22.37 (截距 b=23.177) | Slope=

Process ForkPoolWorker-189:
Process ForkPoolWorker-242:
Process ForkPoolWorker-220:
Process ForkPoolWorker-246:
Process ForkPoolWorker-167:
Process ForkPoolWorker-248:
Process ForkPoolWorker-210:
Process ForkPoolWorker-252:
Process ForkPoolWorker-199:
Process ForkPoolWorker-218:
Process ForkPoolWorker-243:
Process ForkPoolWorker-205:
Process ForkPoolWorker-234:
Process ForkPoolWorker-204:
Process ForkPoolWorker-202:
Process ForkPoolWorker-141:
Process ForkPoolWorker-186:
Process ForkPoolWorker-217:
Process ForkPoolWorker-180:
Process ForkPoolWorker-119:
Process ForkPoolWorker-170:
Process ForkPoolWorker-140:
Process ForkPoolWorker-179:
Process ForkPoolWorker-181:
Process ForkPoolWorker-154:
Process ForkPoolWorker-143:
Process ForkPoolWorker-244:
Process ForkPoolWorker-165:
Process ForkPoolWorker-256:
Process ForkPoolWorker-96:
Process ForkPoolWorker-171:
Process ForkPoolWorker-147:
Process ForkPoolWorker-212:
Process ForkPoolWorker-227:
Process ForkPoolWorker-235:
Process ForkPoolWorke

KeyboardInterrupt: 